In [1]:
import httpx

RERANK_API_KEY = "sk-Q1jDSdlanCNMh7LBlcjo_Q"  # 填写 LiteLLM 网关的 API KEY
RERANK_ENDPOINT = "http://localhost:4000/rerank"  # LiteLLM使用Cohere的兼容接口
RERANK_MODEL = "rerank"

In [2]:
question = "你有没有 A200 的产品图？"

# 教学候选：同型号参数、其他型号图库、同型号视觉素材、同型号售后。
documents = [
    "A200 产品参数：额定功率 200W，支持三档调节，整机重量 1.5kg。",
    "A300 产品图库：包含正面、侧面和使用场景图，可供宣传物料使用。",
    "A200 视觉素材包：包含白底主图、细节特写和场景展示图，可下载用于商品详情页。",
    "A200 售后流程：提交订单号后，客服会协助处理维修和退换货。",
]

In [3]:
async with httpx.AsyncClient(timeout=60) as client:
    response = await client.post(
        RERANK_ENDPOINT,
        headers={"Authorization": f"Bearer {RERANK_API_KEY}"},
        json={
            "model": RERANK_MODEL,
            "query": question,
            "documents": documents,
            "top_n": 4,
        },
    )
    response.raise_for_status()
    payload = response.json()

payload

{'id': 'bcd0220a-6012-996c-bff4-e75322f71d5c',
 'results': [{'index': 2, 'relevance_score': 0.9703577376192567},
  {'index': 0, 'relevance_score': 0.6316372202648817},
  {'index': 1, 'relevance_score': 0.5396126906943787},
  {'index': 3, 'relevance_score': 0.3132529138527357}],
 'meta': {'billed_units': {'total_tokens': 192},
  'tokens': {'input_tokens': 192}}}

In [4]:
from deal_pilot.rag.store.feishu_doc.get_store import get_store

from pymilvus import FunctionType,Function

store = await get_store()

In [7]:
model_reranker = Function(
    name="qwen_rerank",
    input_field_names=["text"],
    function_type=FunctionType.RERANK,
    params={
        "reranker":"model",
        "provider":"cohere",
        "model_name":"rerank",
        "queries":[question]
    }
)

results = await store.asimilarity_search_with_score(
    query=question,
    fetch_k=20,
    k=4,
    reranker =model_reranker
)
[{"doc": doc, "score": score} for doc, score in results]

[{'doc': Document(metadata={'url': 'https://www.feishu.cn/wiki/S1NJwc9raiSxF0kH6fXcP1nBnQe', 'block_type': 27, 'parent_block_id': 'doxcnbIyBp9YX37ACAeFhG5eQlb', 'node_title': 'PROD-001_A200产品与配置说明', 'table_row_index': None, 'obj_token': 'PgJ4dUnbwo1tGzxNvr6c3es9nDb', 'parent_node_token': None, 'order': 6, 'space_id': '7702368561796221957', 'revision_id': 36, 'chunk_index': 0, 'node_token': 'S1NJwc9raiSxF0kH6fXcP1nBnQe', 'chunk_count': 1, 'kind': 'image', 'block_id': 'doxcn8I65oOTAffWVfVUBjoxTUg', 'space_title': '01_产品与配置资料', 'id': 'doxcn8I65oOTAffWVfVUBjoxTUg', 'references': [], 'title_path': ['PROD-001_A200产品与配置说明', 'A200 推理一体机产品与配置说明', '产品外观与内部照片'], 'payload': {'bucket_name': 'feishu-images', 'bucket_filename': 'doxcn8I65oOTAffWVfVUBjoxTUg.png'}}, page_content='# PROD-001_A200产品与配置说明\n\n## A200 推理一体机产品与配置说明\n\n### 产品外观与内部照片\n\n该图片用于展示 A200 推理一体机（两卡版）的正面外观，供销售选型和部件位置核对使用。\n\n图片显示了一台银灰色的机架式服务器设备。前面板左侧设有一个蓝色液晶显示屏，屏幕中显示文字“系统正常”。显示屏下方有两个 USB 接口，每个接口下方均印有白色的 USB 标志。\n\n显示屏右侧贴有一张银色铭牌，铭牌上的文字